In [ ]:
import sys
import glob
import torch

sys.path.append('../utils/')
sys.path.append('../src/model/')

sys.path.append('/om2/user/jmhicks/projects/TextureStreaming/code/')

from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats

from texture_prior.params import model_params, statistics_set, texture_dataset
from texture_prior.utils import path

sys.path.append("/om2/user/bjmedina/auditory-memory/memory/")

from utils.plotting import plot_dprime_by_isi, plot_itemwise_split_half_scatter_df, ensure_dir
from utils.dprime import recompute_dprime_by_isi_per_subject
from utils.reliability import compute_itemwise_split_half_reliability

from utils.loading import load_results, load_results_with_isi0_exclusion, load_results_with_isi0_dprime_exclusion, move_sequences_to_used, load_results_with_exclusion
from utils.loading import load_results_with_exclusion_2
from utils.dprime import recompute_dprime_by_isi_per_subject

from utils.plotting import plot_dprime_by_isi, plot_itemwise_split_half_scatter_df, ensure_dir

from utils.reliability import compute_power_curve
from utils.plotting import plot_power_curve

import DistanceMemoryModel
import encoders



import json
import glob
import sys
import os

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from scipy.stats import norm
from collections import defaultdict


from IPython.display import clear_output

def compute_dprime(hit_rate, fa_rate):
    hit_rate = np.clip(hit_rate, 0.0001, 1 - 0.0001)
    fa_rate = np.clip(fa_rate, 0.0001, 1 - 0.0001)
    return norm.ppf(hit_rate) - norm.ppf(fa_rate)

def recompute_dprime_by_isi(exps, criterion=1):
    hit_counts = defaultdict(int)
    fa_counts  = defaultdict(int)
    signal_counts = defaultdict(int)
    noise_counts  = defaultdict(int)

    for df in exps:
        seen_yt_ids = {}
        yt_ids = df['yt_id'].tolist()
        responses = df['response'].tolist()
        repeats = df['repeat'].tolist()


        for i, (yt, resp, repeat) in enumerate(zip(yt_ids, responses, repeats )):
            if pd.isna(resp) or pd.isna(yt):
                continue

            is_yes = int(int(resp) > criterion)

            if repeat == 'true':
                j = yt_ids[:i].index(yt)
                isi = i - j - 1

                if isi not in [-1,0,1,2,3, 4, 8, 16, 32, 64]:
                    continue

                #print(j, i, isi)
                #print(yt_ids[j], yt)
                signal_counts[isi] += 1
                hit_counts[isi] += is_yes
            elif repeat == 'false':
                noise_counts[-1] += 1  # ISI=-1 for noise trials
                fa_counts[-1] += is_yes

                #seen_yt_ids[yt] = i  # store first appearance

        #print("--")

    # Build results
    results = []
    all_isi_vals = sorted(set(signal_counts) | set(noise_counts))
    for isi in all_isi_vals:
        hits = hit_counts[isi]
        fas  = fa_counts[-1]
        n_signal = signal_counts[isi]
        n_noise  = noise_counts[-1]

        hit_rate = hits / n_signal if n_signal > 0 else np.nan
        fa_rate  = fas  / n_noise  if n_noise  > 0 else np.nan
        dprime_val = compute_dprime(hit_rate, fa_rate) #if np.isfinite(hit_rate) and np.isfinite(fa_rate) else np.nan

        results.append({
            'isi': isi,
            'hits': hits,
            'false_alarms': fas,
            'n_signal': n_signal,
            'n_noise': n_noise,
            'hit_rate': hit_rate,
            'fa_rate': fa_rate,
            'd_prime': dprime_val
        })

    return pd.DataFrame(results).sort_values(by='isi')


def recompute_dprime_by_isi_per_subject(exps, criterion=1):
    allowed_isi = {-1, 0, 1, 2, 3, 4, 8, 16, 32, 64}
    all_results = []

    for subj_idx, df in enumerate(exps):
        hit_counts = defaultdict(int)
        fa_counts  = defaultdict(int)
        signal_counts = defaultdict(int)
        noise_counts  = defaultdict(int)

        yt_ids = df['yt_id'].tolist()
        responses = df['response'].tolist()
        repeats = df['repeat'].tolist()

        for i, (yt, resp, repeat) in enumerate(zip(yt_ids, responses, repeats)):
            if pd.isna(resp) or pd.isna(yt):
                continue

            is_yes = int(int(resp) > criterion)

            if repeat == 'true':
                try:
                    j = yt_ids[:i].index(yt)
                    isi = i - j - 1
                except ValueError:
                    continue  # yt not found in earlier trials

                if isi not in allowed_isi:
                    continue

                signal_counts[isi] += 1
                hit_counts[isi] += is_yes

            elif repeat == 'false':
                isi = -1
                noise_counts[isi] += 1
                fa_counts[isi] += is_yes

        # Aggregate per-ISI results for this subject
        for isi in sorted(signal_counts.keys() | noise_counts.keys()):
            n_signal = signal_counts[isi]
            n_noise  = noise_counts.get(-1, 0)  # all noise trials pooled under -1
            hits = hit_counts[isi]
            fas  = fa_counts.get(-1, 0)

            hit_rate = hits / n_signal if n_signal > 0 else np.nan
            fa_rate  = fas  / n_noise  if n_noise  > 0 else np.nan
            d_prime = compute_dprime(hit_rate, fa_rate) if np.isfinite(hit_rate) and np.isfinite(fa_rate) else np.nan

            all_results.append({
                'subject': subj_idx,
                'isi': isi,
                'hits': hits,
                'false_alarms': fas,
                'n_signal': n_signal,
                'n_noise': n_noise,
                'hit_rate': hit_rate,
                'fa_rate': fa_rate,
                'd_prime': d_prime
            })

    return pd.DataFrame(all_results).sort_values(by=['subject', 'isi'])


def load_results(results_dir, isi_pow=2, min_trials=120, skip_len60=True):
    """Load and filter experiment result CSVs."""
    files = sorted(
        [f for f in os.listdir(results_dir) if f.endswith(".csv")],
        key=lambda fn: os.path.getctime(os.path.join(results_dir, fn))
    )
    exps, seqs, fnames = [], [], []
    for fn in files:
        df = pd.read_csv(os.path.join(results_dir, fn))
        main = df[df.stim_type == "main"]
        seq_file = main.sequence_file.iloc[0].split("/")[-1]
        if len(main) < min_trials: continue
        if "tol0" in seq_file: continue
        exps.append(main); seqs.append(seq_file); fnames.append(fn)
    return exps, seqs, fnames

def remove_sequences_with_len60(seq_dir):
    """Remove entries containing 'len60' from unused.json and used.json."""
    for sub in ("unused", "used"):
        path = os.path.join(seq_dir, sub, f"{sub}.json")
        data = json.load(open(path))
        filtered = [f for f in data if "len60" not in f]
        json.dump(sorted(filtered), open(path, "w"), indent=2)

def move_sequences_to_used(seq_dir, seqs_used):
    """Move used sequence filenames from unused.json to used.json."""
    u_path = os.path.join(seq_dir, "unused", "unused.json")
    z_path = os.path.join(seq_dir, "used",   "used.json")
    unused = json.load(open(u_path)); used = json.load(open(z_path))
    seqs = [os.path.basename(s) for s in seqs_used]
    new_unused = [s for s in unused if s not in seqs]
    new_used = sorted(set(used + seqs))
    json.dump(sorted(new_unused), open(u_path, "w"), indent=2)
    json.dump(new_used,         open(z_path, "w"), indent=2)

def get_dprime_by_isi(df_per_subject, return_sem=False, return_subjects=False):
    """
    Compute mean d-prime per ISI across subjects, excluding ISI = -1 (lures).

    Args:
        df_per_subject (pd.DataFrame): Output from recompute_dprime_by_isi_per_subject.
        return_sem (bool): Whether to return standard error of the mean.
        return_subjects (bool): Whether to return per-subject d-primes too.

    Returns:
        pd.DataFrame or dict:
            If return_sem=False:
                DataFrame with columns ['isi', 'd_prime']
            If return_sem=True:
                DataFrame with columns ['isi', 'd_prime', 'sem']
            If return_subjects=True:
                Returns a dict with:
                    'summary': summary DataFrame as above,
                    'per_subject': filtered per-subject df
    """
    df_filtered = df_per_subject[df_per_subject["isi"] > -1]

    grouped = df_filtered.groupby("isi")["d_prime"]
    result_df = grouped.mean().reset_index(name="d_prime")

    if return_sem:
        result_df["sem"] = grouped.sem().values

    if return_subjects:
        return {
            "summary": result_df,
            "per_subject": df_filtered.copy()
        }

    return result_df.d_prime.tolist()

def plot_groupwise_item_response_scatter(
    results,
    title="Group-wise Item Responses",
    kind="hits",
    seed=42,
    split_method="half",
    save_path=None
):
    """
    Plots a scatter plot of itemwise response rates between two internally split groups.

    Args:
        df (pd.DataFrame): itemwise response matrix (participants x items)
        title (str): plot title
        kind (str): 'hits' or 'false_alarms' for axis labeling
        seed (int): random seed for reproducibility
        split_method (str): 'half' (default) or 'random' to determine splitting method
    """

    r = results['split_half_reliability'][kind][0]

    df = results['itemwise_responses'][kind]
    import matplotlib.pyplot as plt
    if df.shape[0] < 2:
        raise ValueError("DataFrame must have at least 2 participants to split.")

    np.random.seed(seed)

    n = len(df)
    indices = np.arange(n)

    if split_method == "random":
        np.random.shuffle(indices)

    mid = n // 2
    group1_df = df.iloc[indices[:mid]]
    group2_df = df.iloc[indices[mid:]]

    # Ensure columns are aligned
    common_items = sorted(set(group1_df.columns) & set(group2_df.columns))
    if not common_items:
        raise ValueError("No overlapping items between groups.")

    g1_means = group1_df[common_items].mean(axis=0)
    g2_means = group2_df[common_items].mean(axis=0)

    plt.figure(figsize=(6, 6))
    color = "green" if kind == "hits" else "red"
    plt.scatter(g1_means, g2_means, alpha=0.7, color=color, edgecolor="k")

    # Identity line
    lims = [0, 1]
    plt.plot(lims, lims, "--", color="gray", linewidth=1)

    # Optional: correlation
    plt.text(0.05, 0.9, f"r = {r:.2f}", transform=plt.gca().transAxes)

    plt.axis('square')
    plt.xlabel(f"Group 1 {kind} rate")
    plt.ylabel(f"Group 2 {kind} rate")
    plt.title(title)
    plt.xlim(lims)
    plt.ylim(lims)
    plt.gca().set_aspect('equal', adjustable='box')
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300)
        plt.close()
    else:
        plt.show()

def _to_numeric_series(x):
    """Coerce list/Series to numeric pandas Series (NaNs if bad)."""
    return pd.to_numeric(pd.Series(x), errors="coerce")

def _to_binary01_series(x):
    """Coerce responses to {0,1} (True->1, False->0, nonzero->1)."""
    s = pd.Series(x)
    try:
        s = pd.to_numeric(s, errors="coerce")
    except Exception:
        pass
    return s.apply(lambda v: np.nan if pd.isna(v) else int(bool(v)))

# --- Core logic ---
def per_isi_counts(isi_list, responses, noise_value=-1):
    """
    Build counts per ISI level using a shared noise pool at ISI==noise_value.

    Args:
        isi_list (list/Series): ISI values (e.g., -1 for non-repeat, 0, 16 for repeats)
        responses (list/Series): 0/1 (or bool) responses
        noise_value (int/float): value indicating 'non-repeat' trials (default -1)

    Returns:
        pd.DataFrame with columns:
            ['isi', 'n_signal', 'k_hit', 'n_noise', 'k_fa']
        where for each isi != noise_value:
            - n_signal/k_hit come from trials with that isi
            - n_noise/k_fa come from the shared noise pool (isi==noise_value)
    """
    isi = _to_numeric_series(isi_list)
    resp = _to_binary01_series(responses)
    df = pd.DataFrame({"isi": isi, "resp": resp}).dropna(subset=["isi", "resp"])

    # Noise pool (non-repeat trials)
    noise_df = df[df["isi"] == noise_value]
    n_noise = int(len(noise_df))
    k_fa = int((noise_df["resp"] == 1).sum())

    # Signal ISIs (repeat trials)
    signal_levels = (
        df.loc[df["isi"] != noise_value, "isi"]
          .dropna()
          .sort_values()
          .unique()
          .tolist()
    )

    rows = []
    for lvl in signal_levels:
        g = df[df["isi"] == lvl]
        n_sig = int(len(g))
        k_hit = int((g["resp"] == 1).sum())
        rows.append({
            "isi": float(lvl),
            "n_signal": n_sig,
            "k_hit": k_hit,
            "n_noise": n_noise,
            "k_fa": k_fa,
        })

    return pd.DataFrame(rows).sort_values("isi").reset_index(drop=True)

def add_rates_and_dprime(counts_df, edge_correction="loglinear"):
    """
    Convert counts to rates and d'. Supports log-linear edge correction.

    Args:
        counts_df: output of per_isi_counts
        edge_correction: 'loglinear' or None

    Returns:
        counts_df with extra columns: ['hit_rate','fa_rate','dprime']
    """
    out = counts_df.copy()

    if edge_correction == "loglinear":
        # Macmillan & Creelman correction within each category
        hr = (out["k_hit"] + 0.5) / (out["n_signal"] + 1.0)
        far = (out["k_fa"]  + 0.5) / (out["n_noise"]  + 1.0)
    elif edge_correction is None:
        # Raw proportions (clip slightly to avoid inf ppf)
        eps = 0.0001
        hr  = np.clip(out["k_hit"] / out["n_signal"].replace(0, np.nan), eps, 1 - eps)
        far = np.clip(out["k_fa"]  / out["n_noise"].replace(0, np.nan), eps, 1 - eps)
    else:
        raise ValueError("edge_correction must be 'loglinear' or None")

    out["hit_rate"] = hr.astype(float)
    out["fa_rate"]  = far.astype(float)

    # d' = z(HR) - z(FAR)
    out["dprime"] = norm.ppf(out["hit_rate"]) - norm.ppf(out["fa_rate"])
    return out

def compute_per_isi_dprime(isi_list, responses, noise_value=-1, edge_correction="loglinear"):
    """
    One-shot convenience wrapper: from ISI+responses -> per-ISI table with d'.
    """
    counts = per_isi_counts(isi_list, responses, noise_value=noise_value)
    return add_rates_and_dprime(counts, edge_correction=edge_correction)

from scipy.stats import norm

def compute_rates_and_dprime(repeats, responses):
    """
    Compute hit rate, false alarm rate, and d' from lists.
    
    Args:
        repeats (list of str): 'true' or 'false'
        responses (list of int): 0/1 binary responses
    
    Returns:
        hit_rate, fa_rate, d_prime
    """
    df = pd.DataFrame({"repeat": repeats, "resp": responses})
    
    # mask trials
    hits = df[(df.repeat == "true") & (df.resp == 1)]
    misses = df[(df.repeat == "true") & (df.resp == 0)]
    fas = df[(df.repeat == "false") & (df.resp == 1)]
    crs = df[(df.repeat == "false") & (df.resp == 0)]
    
    # rates
    n_signal = len(hits) + len(misses)
    n_noise  = len(fas) + len(crs)
    
    hit_rate = len(hits) / n_signal if n_signal > 0 else np.nan
    fa_rate  = len(fas) / n_noise  if n_noise > 0 else np.nan
    
    # correction to avoid infs
    eps = 0.0001
    hit_rate = np.clip(hit_rate, eps, 1 - eps)
    fa_rate  = np.clip(fa_rate, eps, 1 - eps)
    
    # d'
    dprime = norm.ppf(hit_rate) - norm.ppf(fa_rate)
    return hit_rate, fa_rate, dprime

# Imports at top
import os
import random
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

def compute_itemwise_intergroup_reliability_from_exps(
    expsA,
    expsB,
    *,
    criterionA=1,
    criterionB=1,
    trial_type='hits',          # 'hits' or 'false_alarms'
    min_isi=None,
    max_isi=None,
    SHR_A=None,                 # optional: split-half reliabilities for attenuation
    SHR_B=None,
    n_bootstraps=10000,
    random_seed=42,
    corr_func=pearsonr,
    verbose=False
):
    """
    Intergroup reliability with within-stimulus bootstrap.
    Fixes:
      - Normalizes yt_id via os.path.basename (align paths vs filenames)
      - Robust 2D conversion for per-stimulus rows to prevent 1D arrays
    Returns:
      (bootstrapped_corrs, intergroup_corr, bootstrapped_corrs_attenuated,
       intergroup_corr_attenuated, group_A_true_means, group_B_true_means)
    """
    # RNG setup
    random.seed(random_seed)
    np.random.seed(random_seed)
    rng = np.random.default_rng(random_seed)

    # Normalize flags
    trial_type = str(trial_type).strip().lower()
    if trial_type not in ('hits', 'false_alarms'):
        raise ValueError("trial_type must be 'hits' or 'false_alarms'.")
    exact_isi = (min_isi is not None) and (max_isi is not None) and (min_isi == max_isi)
    isi_target = float(min_isi) if exact_isi else None

    # Row keep predicate (robust repeat/isi handling)
    def _row_is_kept(rep, isi):
        r = str(rep).strip().lower()
        is_hit = r in ('true','1','yes','y','hit','repeat','repeated','t')
        is_fa  = r in ('false','0','no','n','fa','nonrepeat','non-repeat','non','f')
        try:
            isi_val = float(isi)
        except Exception:
            isi_val = np.nan
        if trial_type == 'hits':
            if not is_hit:
                return False
            if exact_isi:
                return np.isfinite(isi_val) and np.isclose(isi_val, isi_target, atol=1e-9)
            if (min_isi is not None) and np.isfinite(isi_val) and isi_val < float(min_isi):
                return False
            if (max_isi is not None) and np.isfinite(isi_val) and isi_val > float(max_isi):
                return False
            return True
        else:
            return is_fa

    # ID normalization (paths -> basename)
    def _norm_id(x):
        try:
            return os.path.basename(str(x).strip())
        except Exception:
            return str(x)

    # Safely convert list-of-[pid, val] to (N,2) float array
    def _to_pairs_2d(rows):
        """
        Ensures a 2D float array with shape (n, 2).
        Handles ragged/object lists and rare single-pair cases.
        """
        try:
            arr = np.array(rows, dtype=float)
            if arr.ndim == 2 and arr.shape[1] == 2:
                return arr
            if arr.ndim == 1:
                # Single pair like [pid, val]
                if len(arr) == 2:
                    return arr.reshape(1, 2).astype(float)
                # Fallback: build from elements
            # Fallback robust path: stack each element as float(2,)
            stacked = np.vstack([np.array(r, dtype=float).reshape(1, 2) for r in rows])
            return stacked.astype(float)
        except Exception:
            stacked = np.vstack([np.array(r, dtype=float).reshape(1, 2) for r in rows])
            return stacked.astype(float)

    # Build responses: stim -> list([pid, response_or_-1])
    def _build_responses_group(exps, criterion):
        nP = len(exps)
        items = set()
        # discover items
        for df in exps:
            if df is None or len(df) == 0:
                continue
            yt_ids    = df['yt_id'].tolist()
            responses = df['response'].tolist()
            repeats   = df['repeat'].tolist()
            isis      = df['isi'].tolist() if 'isi' in df.columns else [np.nan] * len(df)
            for yt, resp, rep, isi in zip(yt_ids, responses, repeats, isis):
                if pd.isna(yt) or pd.isna(resp):
                    continue
                if _row_is_kept(rep, isi):
                    items.add(_norm_id(yt))
        resp_dict = {yt: [[pid, -1] for pid in range(nP)] for yt in items}
        # fill per-participant responses (avg if multiple rows per item)
        for pid, df in enumerate(exps):
            if df is None or len(df) == 0:
                continue
            yt_ids    = df['yt_id'].tolist()
            responses = df['response'].tolist()
            repeats   = df['repeat'].tolist()
            isis      = df['isi'].tolist() if 'isi' in df.columns else [np.nan] * len(df)
            per_item_yeses = {}
            for yt, resp, rep, isi in zip(yt_ids, responses, repeats, isis):
                if pd.isna(yt) or pd.isna(resp):
                    continue
                if not _row_is_kept(rep, isi):
                    continue
                ytn = _norm_id(yt)
                try:
                    is_yes = int(int(resp) > int(criterion))
                except Exception:
                    is_yes = int(float(resp) > float(criterion))
                per_item_yeses.setdefault(ytn, []).append(is_yes)
            for ytn, yes_list in per_item_yeses.items():
                if ytn in resp_dict:
                    resp_dict[ytn][pid][1] = float(np.mean(yes_list))
        return resp_dict

    # Build both groups
    responses_group_A = _build_responses_group(expsA, criterionA)
    responses_group_B = _build_responses_group(expsB, criterionB)

    def _has_valid(rows):
        arr = _to_pairs_2d(rows)                     # shape (n,2), float-able
        resp = arr[:, 1]                             # response column (may be str/obj)
        # safe float coercion (strings -> float, bad values -> NaN)
        resp_num = np.empty_like(resp, dtype=float)
        for i, x in enumerate(resp):
            try:
                resp_num[i] = float(x)
            except Exception:
                resp_num[i] = np.nan
        # valid if any finite response that isn’t the sentinel -1
        return np.any(np.isfinite(resp_num) & (resp_num != -1.0))
    
    # --- build common_stimuli robustly ---
    common_stimuli = []
    for ytn in set(responses_group_A.keys()).intersection(responses_group_B.keys()):
        if _has_valid(responses_group_A[ytn]) and _has_valid(responses_group_B[ytn]):
            common_stimuli.append(ytn)

    if verbose:
        print(f"[intergroup] trial_type={trial_type}, exact_isi={exact_isi}, min_isi={min_isi}, max_isi={max_isi}")
        print(f"[intergroup] items A:{len(responses_group_A)}, B:{len(responses_group_B)}, common:{len(common_stimuli)}")

    if len(common_stimuli) < 2:
        # Not enough overlap to correlate
        return [], np.nan, [], np.nan, [], []

    # Participant indices
    total_number_of_participants_A = len(next(iter(responses_group_A.values())))
    total_number_of_participants_B = len(next(iter(responses_group_B.values())))
    inds_A = np.arange(total_number_of_participants_A)
    inds_B = np.arange(total_number_of_participants_B)

    # Bootstrap
    bootstrapped_corrs = []
    bootstrapped_corrs_attenuated = []

    for _ in range(n_bootstraps):
        group_A_avg, group_B_avg = [], []

        rand_inds_A = rng.choice(inds_A, size=total_number_of_participants_A, replace=True)
        rand_inds_B = rng.choice(inds_B, size=total_number_of_participants_B, replace=True)

        for stim in common_stimuli:
            all_A = _to_pairs_2d(responses_group_A[stim])  # guaranteed (n,2)
            all_A = all_A[np.argsort(all_A[:, 0])]
            resampled_A = all_A[rand_inds_A]
            resampled_A = resampled_A[resampled_A[:, 1] != -1]

            all_B = _to_pairs_2d(responses_group_B[stim])  # guaranteed (n,2)
            all_B = all_B[np.argsort(all_B[:, 0])]
            resampled_B = all_B[rand_inds_B]
            resampled_B = resampled_B[resampled_B[:, 1] != -1]

            if resampled_A.shape[0] == 0 or resampled_B.shape[0] == 0:
                continue

            group_A_avg.append(resampled_A[:, 1].mean())
            group_B_avg.append(resampled_B[:, 1].mean())

        if len(group_A_avg) > 1:
            try:
                r, _ = corr_func(group_A_avg, group_B_avg)
                if not np.isnan(r):
                    bootstrapped_corrs.append(r)
                    if SHR_A is not None and SHR_B is not None:
                        bootstrapped_corrs_attenuated.append(
                            r / np.sqrt(np.mean(SHR_A) * np.mean(SHR_B))
                        )
            except Exception:
                pass

    # "True" (non-resampled) means
    group_A_true, group_B_true = [], []
    for stim in common_stimuli:
        A_arr = _to_pairs_2d(responses_group_A[stim])
        A_arr = A_arr[A_arr[:, 1] != -1]
        B_arr = _to_pairs_2d(responses_group_B[stim])
        B_arr = B_arr[B_arr[:, 1] != -1]
        if A_arr.shape[0] == 0 or B_arr.shape[0] == 0:
            continue
        group_A_true.append(A_arr[:, 1].mean())
        group_B_true.append(B_arr[:, 1].mean())

    if len(group_A_true) > 1:
        intergroup_corr, _ = corr_func(group_A_true, group_B_true)
        intergroup_corr_attenuated = (
            intergroup_corr / np.sqrt(np.mean(SHR_A) * np.mean(SHR_B))
            if SHR_A is not None and SHR_B is not None
            else np.nan
        )
    else:
        intergroup_corr = np.nan
        intergroup_corr_attenuated = np.nan

    return (
        bootstrapped_corrs,
        intergroup_corr,
        bootstrapped_corrs_attenuated,
        intergroup_corr_attenuated,
        group_A_true,
        group_B_true,
    )

def log_results(noise_var, criterion, task, param_idx, r_hits, r_fa, mse):
    """Append one row of results to a CSV file."""
    header = ["param_idx", "task", "noise_var", "criterion", "r_hits", "r_fa", "dprime_mse"]
    row = [param_idx, task, noise_var, criterion, r_hits, r_fa, mse]

    file_exists = os.path.isfile(CSV_PATH)
    with open(CSV_PATH, "a", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(header)
        writer.writerow(row)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
# grabbing example list of sound
sounds_list = glob.glob("/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_p1/*wav")

sounds_list = glob.glob("/mindhive/mcdermott/www/mturk_stimuli/bjmedina/NaturalSounds/*wav")
texture_list = sounds_list

ALL_SOUNDS = glob.glob("/om2/data/public/audioset/wavs/unbalanced_train_segments_downloads/unbalanced_train_segments_downloads_*/*wav")
print(len(ALL_SOUNDS))

tasks = ["ind-nature-len120" ,"global-music-len120", "atexts-len120", "nhs-region-len120"]
which_task = tasks[0] # "global-music-len120", "atexts-len120" "nhs-region-len120"

base_path = "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/{}/sequences/isi_16/len120/"

seqs_paths = {"ind-nature-len120": "mem_exp_ind-nature_2025", 
              "global-music-len120": "global-music-2025-n_80",
              "atexts-len120": "mem_exp_atexts_2025",
              "nhs-region-len120": "nhs-region-n_80"}

hr_task_name = {"ind-nature-len120": "Industrial and Nature", 
              "global-music-len120": "Globalized Music",
              "atexts-len120": "Auditory Textures",
              "nhs-region-len120": " 'Natural History of Song' "}

exps, seqs, fnames = load_results(f"/mindhive/mcdermott/www/bjmedina/experiments/bolivia_2025/results/isi_16/{which_task}")
move_sequences_to_used(base_path.format(seqs_paths[which_task]), seqs)


In [ ]:
skip = False

if skip: 
    
    exps, seqs, fnames, _ = load_results_with_exclusion_2(f"/mindhive/mcdermott/www/bjmedina/experiments/bolivia_2025/results/isi_16/{which_task}",
                                                        min_dprime=2,
                                                        min_trials=120,
                                                        skip_len60=True,
                                                        verbose=False,
                                                        return_skipped=skip)
else:
    exps, seqs, fnames = load_results_with_exclusion_2(f"/mindhive/mcdermott/www/bjmedina/experiments/bolivia_2025/results/isi_16/{which_task}",
                                                    min_dprime=2,
                                                    min_trials=120,
                                                    skip_len60=True,
                                                    verbose=False,
                                                    return_skipped=skip)



move_sequences_to_used(base_path.format(seqs_paths[which_task]), seqs)

print("Number of participants used in analysis:", len(exps))


safe_name = which_task.lower().replace(" ", "_")  # e.g., "globalized_music"
save_dir = os.path.join("/om2/user/bjmedina/auditory-memory/memory/figures/human-results/isi-16-only", safe_name)

# where you want the results
CSV_PATH = f"/om2/user/bjmedina/auditory-memory/memory/assets/lineardistancemodel/{safe_name}/"

ensure_dir(save_dir)
ensure_dir(CSV_PATH)
print(save_dir)

human_results = recompute_dprime_by_isi_per_subject(exps)
human_sensitivity = get_dprime_by_isi(human_results)

clear_output(wait=False)
CSV_PATH = f"/om2/user/bjmedina/auditory-memory/memory/assets/lineardistancemodel/{safe_name}/isi16_runs.csv"

from sklearn.metrics import mean_squared_error

# Your experiment structure (list of stimulus filepaths for each run)
experiment_list = []
for seq in seqs:
    list_to_add = []
    
    with open(base_path.format(seqs_paths[which_task]) + seq, 'r') as file:
        data = json.load(file)
   
    for stim in  data['filenames_order']:
        edited_stim_name = "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/" + seqs_paths[which_task] + "/" + stim
        list_to_add.append(edited_stim_name)
    
    experiment_list.append(list_to_add)

In [ ]:
pc_dims = 256

NV = 0.2                             # per-dim noise std per drift step (your class convention)
CRIT_MULT = 1.5                          # criterion = CRIT_MULT * NV * sqrt(D)
SEED = 123                               # set to None for stochastic runs
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
TIMES_TO_PLOT = [10, 17, 16, 22, 40, 80, 119]   # steps for hist panels

# Map filenames to row indices in X0
# 1) Collect all unique filenames in stable order
seen, all_files = set(), []
for seq in experiment_list:
    for fn in seq:
        if fn not in seen:
            seen.add(fn)
            all_files.append(fn)

name_to_idx = {fn: i for i, fn in enumerate(all_files)}


if which_task == "atexts-len120":

    print("using texture statistiscs")
    pc_texture_model = encoders.AudioTextureEncoderPCA(
        statistics_dict=statistics_set.statistics,
        pc_dims=pc_dims,
        model_params=model_params,
        sr=20000,
        rms_level=0.05,
        duration=2.0,
        device=device
    )
    
    zscore_projector = encoders.ZScoreSpace(pc_texture_model, device=device)
    zscore_projector.fit(sounds_list)

else:
    print("using nn embeddings")

    ARCHITECTURES =  ['kell2018', 'resnet50']
    TASKS = ['word', 'speaker', 'audioset', 'word_speaker_audioset']
    LAYERS = {
        'kell2018': [
            'input_after_preproc', 'relu0', 'relu1', 'relu2', 'relu3', 'relu4', 'relufc'
        ],
        'resnet50': [
            'input_after_preproc', 'conv1_relu1', 'layer1', 'layer2', 'layer3', 'layer4', 'avgpool'
        ],
    }
    
    networks = []
    for architecture in ARCHITECTURES:
        for task in TASKS:
            for layer in LAYERS[architecture]:
                networks.append({'name': f'{architecture}_{task}',
                                'layer': layer})
    
    
    model_name = ARCHITECTURES[0]
    layer = LAYERS[model_name][3]
    task  = TASKS[3]
    
    sys.path.append(f'/om2/user/bjmedina/models/cochdnn/model_directories/{model_name}_{task}/')
    print(f'/om2/user/bjmedina/models/cochdnn/model_directories/{model_name}_{task}/')
    
    
    kell2018_layer = encoders.Kell2018Encoder(
        model_name=model_name,
        layer=layer,
        sr=20000,
        rms_level=0.05,
        duration=2.0,
        device='cuda'
    )

    print("running pca")

    pc_kell = encoders.PCASpace(
        encoder = kell2018_layer,
        n_components = 256, 
        device='cuda'
    )

    pc_kell.fit(sounds_list[:400])
    clear_output(wait=False)

    print("zscoring")

    zscore_projector = encoders.ZScoreSpace(pc_kell, device=device)
    zscore_projector.fit(sounds_list[400:])
    clear_output(wait=False)


In [ ]:
best_noise_var = 0.165
ISI = 16
D = pc_dims
best_criterion = best_noise_var*np.sqrt(D*ISI)

# best_noise_var = 0.4820958945771901
# best_criterion = 79.93584105457306

# best_noise_var = 1.00
# best_criterion = 100
#best_noise_var, best_criterion = 0.165, np.sqrt(120)



memory_model = DistanceMemoryModel.DistanceMemoryModel(
    encoding_model=zscore_projector,
    noise_variance=best_noise_var,
    criterion=best_criterion,
    device='cuda'
)

In [ ]:
from sklearn.metrics import mean_squared_error

# Your experiment structure (list of stimulus filepaths for each run)
experiment_list = []
for seq in seqs:
    list_to_add = []
    
    with open(base_path.format(seqs_paths[which_task]) + seq, 'r') as file:
        data = json.load(file)
   
    for stim in  data['filenames_order']:
        edited_stim_name = "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/" + seqs_paths[which_task] + "/" + stim
        list_to_add.append(edited_stim_name)
    
    experiment_list.append(list_to_add)

In [ ]:
model_dfs = []

# could be that things aren't resetting....
for exp_i, experiment in enumerate(experiment_list):
    df = memory_model.do_experiment(experiment, yt_ids=None, verbose=False)
    # Wrap in a list to simulate a list of subject data
    model_dfs.append(df)
    #memory_model.animate_trials(save_path = f"../movies/tests/distance-model/exp_{exp_i}.gif")
    memory_model.clear_memory()

In [ ]:
# Only consider repeat trials with ISI >= 16
results_long = compute_itemwise_split_half_reliability(exps, min_isi=16, max_isi=16)
hits_humans = results_long['itemwise_responses']['hits']
false_alarms_humans  = results_long['itemwise_responses']['false_alarms']

# Optional coloring
label_hits_fa = {item: 'hit' for item in hits_humans.columns}
label_hits_fa.update({item: 'fa' for item in false_alarms_humans.columns})

print(f"[DEBUG] Final signal df shape: {hits_humans.shape}")
print(f"[DEBUG] Final noise df shape: {false_alarms_humans.shape}")

In [ ]:
# take nanmean of each column
hits_humans = hits_humans.apply(np.nanmean, axis=0)

# # plot as histogram
# plt.xticks([])
# plt.ylim([0,1])
# plt.bar(hits_humans.index, hits_humans.values, alpha=0.8)
# plt.xticks(rotation=90)   # optional, if many stimuli
# plt.ylabel("Mean")
# plt.title("Per-stimulus hit rate (humans)")
# plt.savefig(f"/om2/user/bjmedina/auditory-memory/memory/figures/human-results/isi-16-only/{safe_name}/item-wise-hit-rates.png")

In [ ]:
# take nanmean of each column
false_alarms_humans = false_alarms_humans.apply(np.nanmean, axis=0)

# plot as histogram
# plt.xticks([])
# plt.ylim([0,1])
# plt.bar(false_alarms_humans.index, false_alarms_humans.values, color='r', alpha=0.8)
# plt.xticks(rotation=90)   # optional, if many stimuli
# plt.ylabel("Item-Wise False Alarm Rates")
# plt.title("Per-stimulus Alarm rates (human)")
# plt.savefig(f"/om2/user/bjmedina/auditory-memory/memory/figures/human-results/isi-16-only/{safe_name}/item-wise-false-alarm-rates.png")

In [ ]:
# Only consider repeat trials with ISI >= 16
results_long_model = compute_itemwise_split_half_reliability(model_dfs, criterion=0, min_isi=16, max_isi=16)
hits = results_long_model['itemwise_responses']['hits']
false_alarms  = results_long_model['itemwise_responses']['false_alarms']

# Optional coloring
label_hits_fa = {item: 'hit' for item in hits.columns}
label_hits_fa.update({item: 'fa' for item in false_alarms.columns})

print(results_long_model['split_half_reliability'])
print(f"[DEBUG] Final signal df shape: {hits.shape}")
print(f"[DEBUG] Final noise df shape: {false_alarms.shape}")
print(f"[DEBUG] Columns (signal items): {hits.columns[:5].tolist()}...")
print(f"[DEBUG] Columns (noise items): {false_alarms.columns[:5].tolist()}...")

# Split groups (rows are participants)

safe_name = which_task.lower().replace(" ", "_")  # e.g., "globalized_music"
save_dir = os.path.join("/om2/user/bjmedina/auditory-memory/memory/figures/model-results/distance-model_pc256/n-{}_c-{}/isi-16-only", safe_name)
save_dir_params = save_dir.format(best_noise_var, best_criterion)
ensure_dir(save_dir_params)
hr_cons_plot_dir = os.path.join(save_dir_params, "hit-rate-consistency.png")
far_cons_plot_dir = os.path.join(save_dir_params, "false-alarm-rate-consistency.png")
plot_groupwise_item_response_scatter(results_long_model, kind="hits", save_path=hr_cons_plot_dir)
plot_groupwise_item_response_scatter(results_long_model, kind="false_alarms", save_path=far_cons_plot_dir)
clear_output(wait=False)

In [ ]:
# take nanmean of each column
hits = hits.apply(np.nanmean, axis=0)

# # plot as histogram
# plt.xticks([])
# plt.ylim([0,1])
# plt.bar(hits.index, hits.values)
# plt.xticks(rotation=90)   # optional, if many stimuli
# plt.ylabel("Item-Wise Hit Rates")
# plt.title("Per-stimulus Hit rates (model)")
# #plt.show()
# hrs_plot_dir = os.path.join(save_dir_params, "item-wise-hit-rates.png")

# plt.savefig(hrs_plot_dir)

In [ ]:
# take nanmean of each column
false_alarms = false_alarms.apply(np.nanmean, axis=0)

# # plot as histogram
# plt.xticks([])
# plt.ylim([0,1])
# plt.bar(false_alarms.index, false_alarms.values, color='r',  alpha=0.8)
# plt.xticks(rotation=90)   # optional, if many stimuli
# plt.ylabel("Item-Wise False Alarm Rates")
# plt.title("Per-stimulus False Alarm rates (model)")

# fars_plot_dir = os.path.join(save_dir_params, "item-wise-false-alarm-rates.png")

# plt.savefig(fars_plot_dir)


plt.bar([0,2], [np.mean(false_alarms), np.mean(hits)], alpha=0.5, label="Model")
plt.bar([1,3], [np.mean(false_alarms_humans), np.mean(hits_humans)], alpha=0.5, label="Human")
plt.legend()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 8))  # 2 rows, 1 column

# --- Subplot 1: Histograms ---
axes[0].hist(false_alarms, bins=20, alpha=0.5, color='r', label="Model")
print(np.mean(false_alarms))
axes[0].hist(false_alarms_humans, bins=20, alpha=0.5, color='m', label="Humans")
print(np.mean(false_alarms_humans))

axes[0].set_title("Per-stimulus False Alarm Rates (Model vs Humans)")
axes[0].set_ylabel("Count")
axes[0].set_xlabel("False Alarm Rates")
axes[0].set_xlim([0,1])
axes[0].legend()

# --- Subplot 2: Scatter ---
axes[1].scatter(false_alarms, false_alarms_humans, alpha=0.6, color='purple')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.7)  # identity line for reference
axes[1].set_title("Per-stimulus False Alarm Rates: Model vs Humans")
axes[1].set_xlabel("Model False Alarm Rate")
axes[1].set_ylabel("Human False Alarm Rate")
axes[1].set_ylim([0,1])
axes[1].set_xlim([0,1])
axes[1].set_aspect('equal', adjustable='box')


plt.tight_layout()

# Save
fars_comp_plot_dir = os.path.join(save_dir_params, "item-wise-false-alarm-rates_comparison-humans.png")
plt.savefig(fars_comp_plot_dir, dpi=300)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 8))  # 2 rows, 1 column

# --- Subplot 1: Histograms ---
axes[0].hist(hits, bins=20, alpha=0.5, color='g', label="Model")
#print(np.mean(hits))
axes[0].hist(hits_humans, bins=20, alpha=0.5, color='b', label="Humans")
#print(np.mean(hits_humans))

axes[0].set_title("Per-stimulus Hit Rates (Model vs Humans)")
axes[0].set_ylabel("Count")
axes[0].set_xlabel("Hit Rates")
axes[0].set_xlim([0,1])
axes[0].legend()

# --- Subplot 2: Scatter ---
axes[1].scatter(hits, hits_humans, alpha=0.6, color='purple')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.7)  # identity line for reference
axes[1].set_title("Per-stimulus Hit Rates: Model vs Humans")
axes[1].set_xlabel("Model Hit Rate")
axes[1].set_ylabel("Human Hit Rate")
axes[1].set_ylim([0,1])
axes[1].set_xlim([0,1])
axes[1].set_aspect('equal', adjustable='box')

plt.tight_layout()

# Save
hrs_comp_plot_dir = os.path.join(save_dir_params, "item-wise-hit-rates_comparison-humans.png")
plt.savefig(hrs_comp_plot_dir, dpi=300)
plt.show()

In [ ]:
model_x = recompute_dprime_by_isi_per_subject(model_dfs, criterion=0)
# Assuming `x` is your DataFrame
valid_isi_values = [-1, 0, 16]
model_x_filtered = model_x[model_x['isi'].isin(valid_isi_values)].copy()
model_x = model_x_filtered

isi_plot_dir = os.path.join(save_dir_params, "dprime-vs-isi.png")
print(isi_plot_dir)
plot_dprime_by_isi(model_x, stimulus_set=hr_task_name[which_task], save_path=isi_plot_dir)

In [ ]:
# 1) HITS at exactly ISI=16, human criterion=1, model criterion=0
boot, r, boot_att, r_att, A_means, B_means = compute_itemwise_intergroup_reliability_from_exps(
    expsA=exps,
    expsB=model_dfs,
    criterionA=1,
    criterionB=0,
    trial_type='hits',
    min_isi=16,
    max_isi=16,
    SHR_A=None,            # put arrays of split-half rs here to get attenuation
    SHR_B=None,
    n_bootstraps=5000,
    random_seed=123,
    verbose=True
)

In [ ]:
boot = np.asarray(boot, dtype=float)
boots_h_mean = np.nanmean(np.asarray(boot_fa, dtype=float))

print(boot)

plt.figure(figsize=(7,4))
plt.hist(boot, bins=40, color='steelblue', alpha=0.7)
plt.axvline(r, color='red', linewidth=2, label=f"true r = {r:.3f}")
plt.axvline(np.nanmean(boot), color='black', linestyle='--', label=f"boot mean = {np.nanmean(boot):.3f}")
lo, hi = np.nanpercentile(boot, [2.5, 97.5])
plt.axvline(lo, color='gray', linestyle=':')
plt.axvline(hi, color='gray', linestyle=':', label=f"95% CI = ({lo:.3f}, {hi:.3f})")

plt.title("Intergroup reliability bootstrap distribution (hits, ISI=16)")
plt.xlabel("Pearson correlation r")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
hr_shr_plot_dir = os.path.join(save_dir_params, "hit-rate-intergroup-reliability.png")
plt.savefig(hr_shr_plot_dir)
print(save_dir_params)

In [ ]:
boot_fa, r_fa, boot_att_fa, r_att_fa, A_means_fa, B_means_fa = compute_itemwise_intergroup_reliability_from_exps(
    expsA=exps,
    expsB=model_dfs,
    criterionA=1,
    criterionB=0,
    trial_type='false_alarms',
    min_isi=None,
    max_isi=None,
    SHR_A=None,
    SHR_B=None,
    n_bootstraps=5000,
    random_seed=123,
    verbose=True
)

In [ ]:
import matplotlib.pyplot as plt

boot = np.asarray(boot_fa, dtype=float)
boots_fa_mean = np.nanmean(np.asarray(boot_fa, dtype=float))

plt.figure(figsize=(7,4))
plt.hist(boot, bins=40, color='steelblue', alpha=0.7)
plt.axvline(r_fa, color='red', linewidth=2, label=f"true r = {r_fa:.3f}")
plt.axvline(np.nanmean(boot), color='black', linestyle='--', label=f"boot mean = {np.nanmean(boot):.3f}")
lo, hi = np.nanpercentile(boot, [2.5, 97.5])
plt.axvline(lo, color='gray', linestyle=':')
plt.axvline(hi, color='gray', linestyle=':', label=f"95% CI = ({lo:.3f}, {hi:.3f})")

plt.title("Intergroup reliability bootstrap distribution (false alarms, ISI=16)")
plt.xlabel("Pearson correlation r")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
far_shr_plot_dir = os.path.join(save_dir_params, "false-alarm-rate-intergroup-reliability.png")
plt.savefig(far_shr_plot_dir)

In [ ]:
def log_results(noise_var, criterion, r_hits, r_fa, hit_rate, fa_rate):
    import csv
    """Append one row of results to a CSV file."""
    header = ["noise_var", "criterion", "r_hits", "r_fa", "hit_rate", "fa_rate"]
    row = [noise_var, criterion, r_hits, r_fa, hit_rate, fa_rate]

    file_exists = os.path.isfile(CSV_PATH)
    with open(CSV_PATH, "a", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(header)
        writer.writerow(row)

log_results(best_noise_var, best_criterion, boots_h_mean, boots_fa_mean, np.mean(hits), np.mean(false_alarms))

In [ ]:
# 1) HITS at exactly ISI=16, human criterion=1, model criterion=0
boot, r, boot_att, r_att, A_means, B_means = compute_itemwise_intergroup_reliability_from_exps(
    expsA=exps[:len(exps)//2],
    expsB=model_dfs,
    criterionA=1,
    criterionB=0,
    trial_type='hits',
    min_isi=16,
    max_isi=16,
    SHR_A=None,            # put arrays of split-half rs here to get attenuation
    SHR_B=None,
    n_bootstraps=5000,
    random_seed=123,
    verbose=True
)

boot = np.asarray(boot, dtype=float)
print(boot)

plt.figure(figsize=(7,4))
plt.hist(boot, bins=40, color='steelblue', alpha=0.7)
plt.axvline(r, color='red', linewidth=2, label=f"true r = {r:.3f}")
plt.axvline(np.nanmean(boot), color='black', linestyle='--', label=f"boot mean = {np.nanmean(boot):.3f}")
lo, hi = np.nanpercentile(boot, [2.5, 97.5])
plt.axvline(lo, color='gray', linestyle=':')
plt.axvline(hi, color='gray', linestyle=':', label=f"95% CI = ({lo:.3f}, {hi:.3f})")

plt.title("Intergroup reliability bootstrap distribution (hits, ISI=16)")
plt.xlabel("Pearson correlation r")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
hr_shr_plot_dir = os.path.join(save_dir_params, "hit-rate-intergroup-reliability-first-half.png")
plt.savefig(hr_shr_plot_dir)
print(save_dir_params)

In [ ]:
# 1) HITS at exactly ISI=16, human criterion=1, model criterion=0
boot, r, boot_att, r_att, A_means, B_means = compute_itemwise_intergroup_reliability_from_exps(
    expsA=exps[len(exps)//2:],
    expsB=model_dfs,
    criterionA=1,
    criterionB=0,
    trial_type='hits',
    min_isi=16,
    max_isi=16,
    SHR_A=None,            # put arrays of split-half rs here to get attenuation
    SHR_B=None,
    n_bootstraps=5000,
    random_seed=123,
    verbose=True
)

boot = np.asarray(boot, dtype=float)
print(boot)

plt.figure(figsize=(7,4))
plt.hist(boot, bins=40, color='steelblue', alpha=0.7)
plt.axvline(r, color='red', linewidth=2, label=f"true r = {r:.3f}")
plt.axvline(np.nanmean(boot), color='black', linestyle='--', label=f"boot mean = {np.nanmean(boot):.3f}")
lo, hi = np.nanpercentile(boot, [2.5, 97.5])
plt.axvline(lo, color='gray', linestyle=':')
plt.axvline(hi, color='gray', linestyle=':', label=f"95% CI = ({lo:.3f}, {hi:.3f})")

plt.title("Intergroup reliability bootstrap distribution (hits, ISI=16)")
plt.xlabel("Pearson correlation r")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
hr_shr_plot_dir = os.path.join(save_dir_params, "hit-rate-intergroup-reliability-second-half.png")
plt.savefig(hr_shr_plot_dir)
print(save_dir_params)

In [ ]:
boot_fa, r_fa, boot_att_fa, r_att_fa, A_means_fa, B_means_fa = compute_itemwise_intergroup_reliability_from_exps(
    expsA=exps[:len(exps)//2],
    expsB=model_dfs,
    criterionA=1,
    criterionB=0,
    trial_type='false_alarms',
    min_isi=None,
    max_isi=None,
    SHR_A=None,
    SHR_B=None,
    n_bootstraps=5000,
    random_seed=123,
    verbose=True
)

boot = np.asarray(boot_fa, dtype=float)

plt.figure(figsize=(7,4))
plt.hist(boot, bins=40, color='steelblue', alpha=0.7)
plt.axvline(r_fa, color='red', linewidth=2, label=f"true r = {r_fa:.3f}")
plt.axvline(np.nanmean(boot), color='black', linestyle='--', label=f"boot mean = {np.nanmean(boot):.3f}")
lo, hi = np.nanpercentile(boot, [2.5, 97.5])
plt.axvline(lo, color='gray', linestyle=':')
plt.axvline(hi, color='gray', linestyle=':', label=f"95% CI = ({lo:.3f}, {hi:.3f})")

plt.title("Intergroup reliability bootstrap distribution (false alarms, ISI=16)")
plt.xlabel("Pearson correlation r")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
far_shr_plot_dir = os.path.join(save_dir_params, "false-alarm-rate-intergroup-reliability-first-half.png")
plt.savefig(far_shr_plot_dir)

In [ ]:
boot_fa, r_fa, boot_att_fa, r_att_fa, A_means_fa, B_means_fa = compute_itemwise_intergroup_reliability_from_exps(
    expsA=exps[len(exps)//2:],
    expsB=model_dfs,
    criterionA=1,
    criterionB=0,
    trial_type='false_alarms',
    min_isi=None,
    max_isi=None,
    SHR_A=None,
    SHR_B=None,
    n_bootstraps=5000,
    random_seed=123,
    verbose=True
)

boot = np.asarray(boot_fa, dtype=float)

plt.figure(figsize=(7,4))
plt.hist(boot, bins=40, color='steelblue', alpha=0.7)
plt.axvline(r_fa, color='red', linewidth=2, label=f"true r = {r_fa:.3f}")
plt.axvline(np.nanmean(boot), color='black', linestyle='--', label=f"boot mean = {np.nanmean(boot):.3f}")
lo, hi = np.nanpercentile(boot, [2.5, 97.5])
plt.axvline(lo, color='gray', linestyle=':')
plt.axvline(hi, color='gray', linestyle=':', label=f"95% CI = ({lo:.3f}, {hi:.3f})")

plt.title("Intergroup reliability bootstrap distribution (false alarms, ISI=16)")
plt.xlabel("Pearson correlation r")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
far_shr_plot_dir = os.path.join(save_dir_params, "false-alarm-rate-intergroup-reliability-second-half.png")
plt.savefig(far_shr_plot_dir)